# 31 · Gold: Location, Hazard and Event dimensions
**Owner: Ar** · Layer: gold · Runs after: 21, 24, 30

**What:** build the three remaining dimensions. Each has an **Unknown member (key −1)**, so a fact with a missing value points to Unknown instead of disappearing.

**Surrogate keys are deterministic.** Keys come from a fixed order (`row_number()` over a business attribute), not a random sequence, so every rebuild gives every row the same key. That makes the rebuild test (P10) comparable run to run.

| Dimension | Key rule | Hierarchy for drill-down |
|---|---|---|
| `dim_location` | ABS state code (NSW = 1 … ACT = 8) | Country → state |
| `dim_hazard` | Order of hazard family, then group | Hazard family → hazard group |
| `dim_event` | Order of start date, then event id | — (one row per event) |

This is also where the **scope** is applied: silver events outside the scope are logged to `audit.rejects` with their reason, so silver = gold + exclusions.

In [0]:
%run ./00_common

In [0]:
loc = read("silver", "ref_state").select(F.col("abs_code").alias("location_key"), "state_code", "state_name", "state_type",
                                         F.lit("Australia").alias("country"))
n_loc = write(with_unknown(loc, location_key=-1, state_code="UNK", state_name="Unknown", state_type="Unknown", country="Unknown"),
              "gold", "dim_location")

hz = read("silver", "xw_hazard").select("hazard_group", "hazard_family").distinct()
hz = hz.withColumn("hazard_key", F.row_number().over(Window.orderBy("hazard_family", "hazard_group"))).select("hazard_key", "hazard_group", "hazard_family")
n_hz = write(with_unknown(hz, hazard_key=-1, hazard_group="Unknown", hazard_family="Unknown"), "gold", "dim_hazard")
log_step("gold", "build_dim_location", "gold.dim_location", n_loc, n_loc)
log_step("gold", "build_dim_hazard", "gold.dim_hazard", n_hz, n_hz)

### Scope: log exclusions, then build the event dimension

In [0]:
ev = read("silver", "ica_events")
rows_in = ev.count()
excluded = ev.where("NOT in_scope").withColumnRenamed("source_row", "_row_num")
n_excl = reject(excluded, "gold", "ica", "SCOPE", F.col("exclusion_reason"))

match = read("silver", "ica_de_match").select("event_id", "de_id", "match_confidence")
dim_event = (ev.where("in_scope").join(match, "event_id", "left")
             .withColumn("event_key", F.row_number().over(Window.orderBy("start_date", "event_id")))
             .select("event_key", "event_id", "cat_code", "declaration_status", "event_name", "hazard_type_raw",
                     "state_raw", "all_states", "state_source", "multi_state_flag", "state_name_conflict",
                     "date_century_fixed", "end_before_start", "claims_parts_mismatch",
                     F.col("de_id").isNotNull().alias("de_matched"), "de_id", "match_confidence",
                     "towns", "description"))
n_ev = write(with_unknown(dim_event, event_key=-1, event_id="UNKNOWN", event_name="Unknown", state_source="unknown"), "gold", "dim_event")
log_step("gold", "build_dim_event", "gold.dim_event", rows_in, n_ev - 1, n_excl, "rows_out excludes the Unknown row; rejected = out of scope")

In [0]:
display(read("gold", "dim_location"))
display(read("gold", "dim_hazard").orderBy("hazard_key"))
display(read("audit", "rejects").where(f"run_id = '{RUN_ID}' AND layer = 'gold'").groupBy("reason").count())